# NCA 1D Dataset Visualization

Visualize datasets generated by `data/build_nca1d_dataset.py`.

- Each NCA state is generated as `H x 1`
- Time steps are concatenated along width to form an `H x T` image
- Stored arrays follow the same `PuzzleDataset`-compatible format as the Sudoku builder

In [1]:
import importlib
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists() and (REPO_ROOT.parent / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import data.build_nca1d_dataset as build_nca1d_dataset
from data.common import PuzzleDatasetMetadata

build_nca1d_dataset = importlib.reload(build_nca1d_dataset)

DATA_DIR = Path(os.environ.get("NCA1D_DATA_DIR", str(REPO_ROOT / "data" / "nca1d-default")))
print("DATA_DIR:", DATA_DIR)


DATA_DIR: /workspaces/URM/data/nca1d-default


In [2]:
with open(DATA_DIR / "config.json") as f:
    builder_config = json.load(f)

def load_optional_array(path: Path):
    return np.load(path) if path.exists() else None

def load_split(split: str):
    split_dir = DATA_DIR / split
    with open(split_dir / "dataset.json") as f:
        metadata = PuzzleDatasetMetadata(**json.load(f))

    dataset = {
        "inputs": np.load(split_dir / "all__inputs.npy", mmap_mode="r"),
        "labels": np.load(split_dir / "all__labels.npy", mmap_mode="r"),
        "puzzle_identifiers": np.load(split_dir / "all__puzzle_identifiers.npy"),
        "puzzle_indices": np.load(split_dir / "all__puzzle_indices.npy"),
        "group_indices": np.load(split_dir / "all__group_indices.npy"),
        "gzip_ratio": np.load(split_dir / "all__gzip_ratio.npy"),
        "state_heights": load_optional_array(split_dir / "all__state_heights.npy"),
        "num_frames": load_optional_array(split_dir / "all__num_frames.npy"),
        "rollout_steps": load_optional_array(split_dir / "all__rollout_steps.npy"),
    }
    return metadata, dataset

train_meta, train_data = load_split("train")
test_meta, test_data = load_split("test")

print("Builder config:")
print(json.dumps(builder_config, indent=2))
print()
print("Train metadata:")
print(train_meta.model_dump())
print()
print("Test metadata:")
print(test_meta.model_dump())


Builder config:
{
  "output_dir": "data/nca1d-default",
  "train_size": 1024,
  "test_size": 256,
  "seed": 0,
  "state_height": 16,
  "state_width": 1,
  "num_colors": 8,
  "temperature": 0.001,
  "identity_bias": 0.0,
  "conv_channels": 4,
  "hidden_dim": 16,
  "patch_size": 1,
  "rollout_steps": 64,
  "time_subsample": 1,
  "start_step": 0,
  "gzip_threshold_low": null,
  "gzip_threshold_high": null,
  "batch_candidate_size": 64,
  "max_sampling_rounds": 200,
  "token_offset": 2,
  "save_dtype": "int32",
  "num_frames": 64,
  "seq_len": 1024,
  "vocab_size": 10,
  "final_image_shape": [
    16,
    64
  ],
  "time_axis": "width",
  "value_encoding": {
    "pad": 0,
    "mask": 1,
    "nca_color_range": [
      2,
      9
    ]
  }
}

Train metadata:
{'pad_id': 0, 'ignore_label_id': 0, 'blank_identifier_id': 0, 'vocab_size': 10, 'seq_len': 1024, 'num_puzzle_identifiers': 1, 'total_groups': 1024, 'mean_puzzle_examples': 1.0, 'sets': ['all']}

Test metadata:
{'pad_id': 0, 'ignore_label

In [3]:
PADDED_HEIGHT = int(builder_config.get("resolved_state_height_max", builder_config["state_height"]))
PADDED_FRAMES = int(builder_config.get("max_num_frames", builder_config["num_frames"]))
TOKEN_OFFSET = int(builder_config["token_offset"])
NUM_COLORS = int(builder_config["num_colors"])

def sample_height(dataset, sample_idx: int) -> int:
    if dataset["state_heights"] is None:
        return PADDED_HEIGHT
    return int(dataset["state_heights"][sample_idx])

def sample_num_frames(dataset, sample_idx: int) -> int:
    if dataset["num_frames"] is None:
        return PADDED_FRAMES
    return int(dataset["num_frames"][sample_idx])

def decode_image(dataset, sample_idx: int) -> np.ndarray:
    image_height = sample_height(dataset, sample_idx)
    num_frames = sample_num_frames(dataset, sample_idx)
    return build_nca1d_dataset.unflatten_time_image(
        np.asarray(dataset["inputs"][sample_idx]),
        image_height=image_height,
        num_frames=num_frames,
        token_offset=TOKEN_OFFSET,
        padded_height=PADDED_HEIGHT,
        padded_num_frames=PADDED_FRAMES,
    )

print("Train arrays:")
for key, value in train_data.items():
    print(f"  {key:18s} shape={value.shape} dtype={value.dtype}")
print()
print("Test arrays:")
for key, value in test_data.items():
    print(f"  {key:18s} shape={value.shape} dtype={value.dtype}")
print()
print("Padded image shape:", (PADDED_HEIGHT, PADDED_FRAMES))
print("Decoded image shape:", decode_image(train_data, 0).shape)
print("Value range after decode:", int(decode_image(train_data, 0).min()), int(decode_image(train_data, 0).max()))
if train_data["state_heights"] is not None:
    print("Train sampled heights:", int(train_data["state_heights"].min()), int(train_data["state_heights"].max()))
if train_data["num_frames"] is not None:
    print("Train sampled frame counts:", int(train_data["num_frames"].min()), int(train_data["num_frames"].max()))


Train arrays:
  inputs             shape=(1024, 1024) dtype=int32
  labels             shape=(1024, 1024) dtype=int32
  puzzle_identifiers shape=(1024,) dtype=int32
  puzzle_indices     shape=(1025,) dtype=int32
  group_indices      shape=(1025,) dtype=int32
  gzip_ratio         shape=(1024,) dtype=float32


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
def show_examples(dataset, split_name: str, num_examples: int = 6, seed: int = 0):
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset["inputs"]), size=min(num_examples, len(dataset["inputs"])), replace=False)
    cmap = plt.get_cmap("tab20", NUM_COLORS)

    fig, axes = plt.subplots(len(indices), 1, figsize=(12, 2.0 * len(indices)), squeeze=False)
    for row_idx, sample_idx in enumerate(indices):
        image = decode_image(dataset, sample_idx)
        ax = axes[row_idx, 0]
        ax.imshow(image, cmap=cmap, interpolation="nearest", aspect="auto", vmin=0, vmax=NUM_COLORS - 1)
        ax.set_title(
            f"{split_name} sample {sample_idx} | shape={image.shape[0]}x{image.shape[1]} | gzip={dataset['gzip_ratio'][sample_idx]:.4f}"
        )
        ax.set_ylabel("state")
        ax.set_xlabel("time")
    plt.tight_layout()
    plt.show()

show_examples(train_data, "train", num_examples=6, seed=0)
show_examples(test_data, "test", num_examples=4, seed=1)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_data["gzip_ratio"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Train gzip ratio")
axes[0].set_xlabel("compressed / raw")
axes[0].set_ylabel("count")

axes[1].hist(test_data["gzip_ratio"], bins=20, color="darkorange", edgecolor="white")
axes[1].set_title("Test gzip ratio")
axes[1].set_xlabel("compressed / raw")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

if train_data["state_heights"] is not None and train_data["num_frames"] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(train_data["state_heights"], bins=20, color="mediumpurple", edgecolor="white")
    axes[0].set_title("Train sampled heights")
    axes[0].set_xlabel("height")
    axes[0].set_ylabel("count")
    axes[1].hist(train_data["num_frames"], bins=20, color="seagreen", edgecolor="white")
    axes[1].set_title("Train sampled frame counts")
    axes[1].set_xlabel("frames")
    axes[1].set_ylabel("count")
    plt.tight_layout()
    plt.show()

train_colors = np.concatenate([decode_image(train_data, idx).reshape(-1) for idx in range(min(64, len(train_data['inputs'])))], axis=0)
unique, counts = np.unique(train_colors, return_counts=True)
print("Sampled train color histogram:")
for value, count in zip(unique, counts):
    print(f"  color={int(value):2d} count={int(count):6d}")
